# Disciplina: Teoria de Grafos

- Matriz de adjacências
- Número de vértices n
- Número de arestas e
- Grau de vértices d(G)

### Prof. Marcos W. Rodrigues


In [6]:
import numpy as np
import pandas as pd
from collections import defaultdict
import csv

In [7]:
folder = "graphs/"

In [238]:
class Graph(object):


    """
    Initiate the graph with an adjancency matrix read from a csv file.

    Parameters
    ----------
    file : str
        The name of the csv file containing the adjacency matrix.
    """
    def __init__(self, file):
        df = pd.read_csv(folder+file, sep=';', header=0, engine='python')
        matrix = df.values
        
        self.n = len(matrix)

        self.init_AdjList(matrix)

    """
    Initialize the adjacency list.

    Parameters
    ----------
    matrix : list of list of int
        The adjacency matrix of the graph.
    """
    def init_AdjList(self,matrix):
        self.adjList = defaultdict(list)

        for i in range(self.n):
            for j in range(self.n):
                if matrix[i][j] == 1:
                    self.adjList[i].append(j)

        self.validateGraph()

    def validateGraph(self):
        for v, adj in self.adjList.items():
            for u in adj:
                if v not in self.adjList[u]:
                    raise ValueError(f"Edge ({v}, {u}) is not symmetric.")
        
        #handshake lemma: the number of vertices with odd degree must be even
        if sum(1 for d in self.nodesDegrees() if d % 2 == 1) % 2 != 0:
            raise ValueError("Invalid graph: odd number of odd-degree vertices.")

    """
    Return the number of vertices in the graph.

    Returns
    -------
    int
        The number of vertices in the graph.
    """
    def __len__(self):
        return self.n

    """
    Add an edge between vertices u and v.
    
    Parameters
    ----------
    u : int
        The first vertex.
    v : int
        The second vertex.
    """
    def add_edge(self, u, v):

        if not (0 <= u < self.n and 0 <= v < self.n):
            raise ValueError("node out of bounds")

        self.adjList[u].append(v)
        self.adjList[v].append(u)

        self.validateGraph()

    """
    Remove the edge between vertices u and v.
    
    Parameters
    ----------
    u : int
        The first vertex.
    v : int
        The second vertex.
    """
    def remove_edge(self, u, v):
        self.adjList[u].remove(v)
        self.adjList[v].remove(u)

        self.validateGraph()
    
    """
    Return the number of Vertices in the graph.

    Returns
    -------

    int
        The number of vertices in the graph.
    """
    def numNodes(self):
        return self.n
    
    """
    Return the number of edges in the graph.
    """
    def numEdges(self):
        return(int (sum(self.nodesDegrees()) / 2))
    
    """
    Return a dictionary with the degree of each vertex in the graph.
    
    Returns:
    -------
    dict
        A dictionary with the degree of each vertex in the graph.
    """
    def nodesDegrees(self):
        degrees = {        }

        degrees = [len(v) for v in self.adjList.values()]

        return degrees
    
    """
    Return True if the graph has parallel edges, False otherwise.
    
    Returns:
    -------
    bool        
        True if the graph has parallel edges, False otherwise. 
    """
    def hasParallelEdges(self):
        if any(len(adj) != len(set(adj)) for adj in self.adjList.values()):
            return True
        return False
    
    """
    Return a dictionary with the parallel edges in the graph.

    Returns:
    -------
    dict
        A dictionary with the parallel edges in the graph.
    """
    def parallelEdges(self):
        parallelEdges = {}
        for v in self.adjList:
            if len(self.adjList[v]) != len(set(self.adjList[v])) and v not in self.adjList[v]:
                parallelEdges[v] = self.adjList[v]
        return parallelEdges
    
    def hasLoopEdges(self):
        return self.loopEdges() != {}

    def loopEdges(self):
        loopEdges = {
            v: adj 
            for v, adj in self.adjList.items() 
            if v in adj
            }
        return loopEdges
    
    def nodeDegree(self, v):
        return len(self.adjList[v])
    
    def printAdjList(self):
        for v in self.adjList:
            print(f"Vertex {v}: {self.adjList[v]}")

    def isSimple(self):
         return not self.hasParallelEdges() and not self.hasloopEdges()
    
    def isNull(self):
        if self.numEdges() == 0:
            return True
        return False
    
    def isRegular(self):
        if not self.adjList:
            return True

        dG_0 = len(next(iter(self.adjList.values())))

        for neighbors in self.adjList.values():
            if len(neighbors) != dG_0:
                return False
            
        return True
    
    def isolatedNodes(self):
        return self.nodesOfDegree(0)
    
    def pendingNodes(self):
        return self.nodesOfDegree(1)
    
    def nodesOfDegree(self, degree):
        return [v for v in range(self.n) if self.nodeDegree(v) == degree]
        
    def isComplete(self):
        return self.numEdges() == (self.n - 1) * self.n / 2 

    def isConnected(self):  
        
        bfs_visited, _, _ = self.bfs(0)

        return len(bfs_visited) == self.n

    def isCircular(self):

        if self.n == 1 and self.hasLoopEdges(): return True

        if self.n < 3:
            return False

        if self.isSimple() and self.isConnected() and self.numEdges() == self.numNodes():
            return True

        return False
    
    """
    INCOMPLETE
    Return True if the graph and another graph G2 are isomorphs, False otherwise.

    Parameters
    ----------
    G2 : Graph
        The graph to compare with.
    
    Returns
    -------
    bool
        True if the graph and G2 are isomorphs, False otherwise.
    """
    def areIsomorphs(self, G2):
        if self.numNodes() != G2.numNodes() or self.numEdges() != G2.numEdges():
            return False
    
    def areComplementary(self, G2):
        return( "not implemented")
    
    """
    Breadth First Search (BFS) Implementation

    Parameters
    ----------
    root : int
        The starting vertex for the BFS.
    
    Returns
    -------
    visited : set
        A set of visited vertices.
    parent : dict
        A dictionary mapping each vertex to its parent in the BFS tree.
    dist : dict
        A dictionary mapping each vertex to its distance from the root vertex.
    """
    def bfs(self, root):
        visited = set()
        parent = {root: None}
        dist = {root: 0}

        queue = [root]

        while len(queue) > 0:
            v = queue.pop(0)
            
            if v not in visited:
                visited.add(v)

                for w in self.adjList[v]:
                    if w not in dist:
                        dist[w] = dist[v] + 1
                        parent[w] = v
                        queue.append(w)
        
        return visited, parent, dist

In [245]:
n3e2 = Graph("n3e2(in).csv")
print("n =", n3e2.numNodes(), "\te =", n3e2.numEdges() )

n = 3 	e = 2


In [254]:
print("loop {}".format(n3e2.loopEdges()))
print("paralel {}".format(n3e2.parallelEdges()))
n3e2.adjList.items()

loop {1: [1, 1, 2]}
paralel {}


dict_items([(0, [2]), (1, [1, 1, 2]), (2, [0, 1])])

In [219]:
K5 = Graph("K5(in).csv")
print("n =", K5.numNodes(), "\te =", K5.numEdges() )

n = 5 	e = 10


In [220]:
K5.printAdjList()
K5.isComplete(
    
)

Vertex 0: [1, 2, 3, 4]
Vertex 1: [0, 2, 3, 4]
Vertex 2: [0, 1, 3, 4]
Vertex 3: [0, 1, 2, 4]
Vertex 4: [0, 1, 2, 3]


True

In [27]:
heap = Graph("heap.csv")

In [29]:
heap.printAdjList()

[1, 2, 3]
[0]
[0]
[0, 4, 5]
[3]
[3, 4]


In [30]:
e = heap.numEdges()
print("Numero de arestas:", e)

Numero de arestas: 5
